AI Assistance: OpenAI ChatGPT and Anthropic's Claude were used for code debugging, code generation, code organization, and code methodological brainstorming\. All final modeling, implementation, validation, commentary, and interpretation were performed and verified by the authors\.

## Project Icarus Data Extraction

For this project, we utilized a variety of data sources:
- Aurorasaurus
   - https://zenodo.org/records/16783265?preview_file=web_observations_2014-08-01_to_2025-08-02_cleaned.csv
   - The link above contains observations from 2014 to 2025.
   - A HUGE limitation is that this is human observation data, so it is not perfect or completely accurate.
- NASA OMNIWEB
   - https://omniweb.gsfc.nasa.gov/form/dx1.html
   - As of 7/4, these variables were selected. Of course, you are free to change them, just update the files here!
```
    ITEMS                      FORMAT   
     
 1 Year                          I4        
 2 Day                           I4        
 3 Hour                          I3        
 4 Minute                        I3        
 5 Field magnitude average, nT   F8.2      
 6 BX, nT (GSE, GSM)             F8.2      
 7 BY, nT (GSM)                  F8.2      
 8 BZ, nT (GSM)                  F8.2      
 9 Speed, km/s                   F8.1      
10 Proton Density, n/cc          F7.2     

```


# Imports

In [1]:
import pandas as pd
import numpy as np
import requests
from io import StringIO
import re

# Cleaning up Aurorasaurus Data

In [2]:
aurora_df = pd.read_csv("data/Aurorasaurus/aurorasaurus_observations_2014-2025.csv")
print("Columns:", aurora_df.columns)
aurora_df.head()

Columns: Index(['raw_row_num', 'obs_id', 'activities_id', 'height_id', 'sky_id',
       'observer_id', 'timestamp', 'address_country', 'address_state',
       'location', 'see_aurora', 'sky_other', 'time_start', 'time_end',
       'on_going', 'height_other', 'activities_other', 'colors_other',
       'types_other', 'comment', 'image', 'st_y', 'st_x', 'colors', 'types',
       'image_url', 'small_image_url', 'redact_row', 'pii_redacted',
       'duration', 'mlat', 'mlon', 'mlt'],
      dtype='object')


,raw_row_num,obs_id,activities_id,height_id,sky_id,observer_id,timestamp,address_country,address_state,location,...,colors,types,image_url,small_image_url,redact_row,pii_redacted,duration,mlat,mlon,mlt
0,1,7372c36fe0c7,NaN,NaN,NaN,117.0,2014-10-01 08:21:34.818216+00:00,United Arab Emirates,Dubai,SRID=4326;POINT (55.154846755193745 25.0416718...,...,NaN,NaN,NaN,NaN,False,False,0.25,19.985358,128.137506,12.129542
1,3,07c100d1ef09,NaN,NaN,clea,NaN,2014-10-02 13:47:29.552583+00:00,United States,NM,SRID=4326;POINT (-106.31864826448822 35.874131...,...,NaN,NaN,NaN,NaN,False,False,0.25,44.155324,-39.100305,6.438991
2,6,90f6d683296b,quie,n,NaN,140.0,2014-10-08 18:09:23.114761+00:00,United States,NM,SRID=4326;POINT (-106.0875978020242 35.9777940...,...,"red, whit",arcs,NaN,NaN,False,False,0.25,44.298471,-38.831876,10.782292
3,7,e6ffd03f4085,very,n45,NaN,141.0,2014-10-08 18:11:38.395952+00:00,United States,NM,SRID=4326;POINT (-106.33536318032095 35.858267...,...,gree,"glow, patc",NaN,NaN,False,False,0.25,44.136071,-39.115849,10.763361
4,8,a67afd419547,acti,over,NaN,141.0,2014-10-08 18:15:51.473713+00:00,United States,NM,SRID=4326;POINT (-106.3362134605038 35.8694066...,...,"whit, gree, pink","arcs, glow",NaN,NaN,False,False,0.25,44.147181,-39.118361,10.763193


In [3]:
# A lot of this data we can disregard, such as identifiers, location (redundant with st_x, and st_y), and others
# We are mainly interested in see_aurora

columns = ["timestamp", "st_y", "st_x", "mlat", "mlon", "mlt", "see_aurora"]
cleaned_aurora_df = aurora_df[columns]
cleaned_aurora_df = cleaned_aurora_df.rename(
    columns={"st_y": "lat", 
             "st_x": "lon",
             "mlat": "mag_lat",
             "mlon": "mag_lon",
             "mlt": "mag_lt",
             "see_aurora": "aurora_observed"}
    )
cleaned_aurora_df.head()

,timestamp,lat,lon,mag_lat,mag_lon,mag_lt,aurora_observed
0,2014-10-01 08:21:34.818216+00:00,25.041672,55.154847,19.985358,128.137506,12.129542,False
1,2014-10-02 13:47:29.552583+00:00,35.874132,-106.318648,44.155324,-39.100305,6.438991,False
2,2014-10-08 18:09:23.114761+00:00,35.977794,-106.087598,44.298471,-38.831876,10.782292,True
3,2014-10-08 18:11:38.395952+00:00,35.858267,-106.335363,44.136071,-39.115849,10.763361,True
4,2014-10-08 18:15:51.473713+00:00,35.869407,-106.336213,44.147181,-39.118361,10.763193,True


In [4]:
cleaned_aurora_df.isna().sum()

timestamp           0
lat                 0
lon                 0
mag_lat            16
mag_lon            16
mag_lt             16
aurora_observed     0
dtype: int64

In [5]:
cleaned_aurora_df = cleaned_aurora_df.dropna()
len(cleaned_aurora_df)

22264

In [6]:
cleaned_aurora_df.to_csv("data/Processed/aurora_clean.csv", index=False)

# Pulling from OMNIWEB

In [7]:
# Info pulled from https://omniweb.gsfc.nasa.gov/html/omni_min_data.html
OMNIWEB_VARS = {
    "mag_avg_nt": 13,
    "bx_gsm_nt": 14,
    "by_gsm_nt": 17,
    "bz_gsm_nt": 18,
    "flow_speed_km_s": 21,
    "proton_density_n_cc": 25,
}
na_values = [99999.9, 9999.99, 999.99]

var_params = ""
columns = ["year","day","hour","minute"]
for var, id in OMNIWEB_VARS.items():
    var_params += f"vars={id}&"
    columns.append(var)

In [8]:
def get_omniweb_year(start_date, end_date):
    url = f"https://omniweb.gsfc.nasa.gov/cgi/nx1.cgi?activity=retrieve&res=min&spacecraft=omni_min&start_date={start_date}&end_date={end_date}&{var_params}"
    response = requests.get(url)
    text = response.text
    lines = text.splitlines()

    # Header
    for i, line in enumerate(lines):
        if line.startswith("YYYY DOY HR MN"):
            header_idx = i
            break

    data_lines = [
        line for line in lines[header_idx + 1:]
        if re.match(r"^\d{4}\s", line)
    ]

    df = pd.read_csv(
        StringIO("\n".join(data_lines)),
        sep=r"\s+",
        names=columns,
        na_values=na_values
    )
    return df.copy()

In [9]:
omni_df = pd.DataFrame(columns=columns)
for i in range(2000, 2026):
    start_date = f"{i}0101"
    end_date = f"{i}1231"
    print("Pulling data from year", i, "...")
    df = get_omniweb_year(start_date, end_date)
    omni_df = pd.concat([omni_df, df], ignore_index=True)
print("Complete!")

Pulling data from year 2000 ...
/tmp/ipykernel_96/941071674.py:12: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  omni_df = pd.concat([omni_df, df], ignore_index=True)
Pulling data from year 2001 ...
Pulling data from year 2002 ...
Pulling data from year 2003 ...
Pulling data from year 2004 ...
Pulling data from year 2005 ...
Pulling data from year 2006 ...
Pulling data from year 2007 ...
Pulling data from year 2008 ...
Pulling data from year 2009 ...
Pulling data from year 2010 ...
Pulling data from year 2011 ...
Pulling data from year 2012 ...
Pulling data from year 2013 ...
Pulling data from year 2014 ...
Pulling data from year 2015 ...
Pulling data from year 2016 ...
Pulling data from year 2017 ...
Pulling data from year 2018 ...
Pulling

In [10]:
# Most up to date for minute-data (as of 7/17/2026), is up to 06/30/2026
final_start = "20260101"
final_end = "20260630"
omni_df = pd.concat([omni_df, get_omniweb_year(final_start, final_end)], ignore_index=True)

In [11]:
omni_df

,year,day,hour,minute,mag_avg_nt,bx_gsm_nt,by_gsm_nt,bz_gsm_nt,flow_speed_km_s,proton_density_n_cc
0,2000,1,0,0,6.82,-5.94,0.27,-0.08,NaN,NaN
1,2000,1,0,1,6.99,-5.88,1.95,1.08,664.7,3.12
2,2000,1,0,2,6.99,-5.71,2.74,2.24,663.2,3.24
3,2000,1,0,3,6.83,-5.33,3.18,2.78,662.2,3.11
4,2000,1,0,4,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
13936315,2026,181,23,55,17.93,0.28,-16.97,5.79,432.8,18.85
13936316,2026,181,23,56,18.15,0.01,-17.13,6.01,432.3,16.93
13936317,2026,181,23,57,18.12,0.16,-16.98,6.31,432.1,16.35
13936318,2026,181,23,58,17.98,0.36,-16.55,7.02,431.1,17.80


In [12]:
omni_df["day"] = omni_df['day'].astype('int64')
omni_df["hour"] = omni_df['hour'].astype('int64')
omni_df["minute"] = omni_df['minute'].astype('int64')

omni_df["day_cos"] = np.cos(2*np.pi * (omni_df["day"]/365.0))
omni_df["day_sin"] = np.sin(2*np.pi * (omni_df["day"]/365.0))
omni_df["hour_cos"] = np.cos(2*np.pi * (omni_df["hour"]/24.0))
omni_df["hour_sin"] = np.sin(2*np.pi * (omni_df["hour"]/24.0))
omni_df["minute_cos"] = np.cos(2*np.pi * (omni_df["minute"]/60.0))
omni_df["minute_sin"] = np.sin(2*np.pi * (omni_df["minute"]/60.0))
omni_df

,year,day,hour,minute,mag_avg_nt,bx_gsm_nt,by_gsm_nt,bz_gsm_nt,flow_speed_km_s,proton_density_n_cc,day_cos,day_sin,hour_cos,hour_sin,minute_cos,minute_sin
0,2000,1,0,0,6.82,-5.94,0.27,-0.08,NaN,NaN,0.999852,0.017213,1.000000,0.000000,1.000000,0.000000
1,2000,1,0,1,6.99,-5.88,1.95,1.08,664.7,3.12,0.999852,0.017213,1.000000,0.000000,0.994522,0.104528
2,2000,1,0,2,6.99,-5.71,2.74,2.24,663.2,3.24,0.999852,0.017213,1.000000,0.000000,0.978148,0.207912
3,2000,1,0,3,6.83,-5.33,3.18,2.78,662.2,3.11,0.999852,0.017213,1.000000,0.000000,0.951057,0.309017
4,2000,1,0,4,NaN,NaN,NaN,NaN,NaN,NaN,0.999852,0.017213,1.000000,0.000000,0.913545,0.406737
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13936315,2026,181,23,55,17.93,0.28,-16.97,5.79,432.8,18.85,-0.999667,0.025818,0.965926,-0.258819,0.866025,-0.500000
13936316,2026,181,23,56,18.15,0.01,-17.13,6.01,432.3,16.93,-0.999667,0.025818,0.965926,-0.258819,0.913545,-0.406737
13936317,2026,181,23,57,18.12,0.16,-16.98,6.31,432.1,16.35,-0.999667,0.025818,0.965926,-0.258819,0.951057,-0.309017
13936318,2026,181,23,58,17.98,0.36,-16.55,7.02,431.1,17.80,-0.999667,0.025818,0.965926,-0.258819,0.978148,-0.207912


In [13]:
kp_df = pd.read_csv('data/OMNIWEB/omni_kp_20000101_20260630.lst', sep=r'\s+')
kp_df

,year,day,hour,kp_10
0,2000,1,0,53
1,2000,1,1,53
2,2000,1,2,53
3,2000,1,3,47
4,2000,1,4,47
...,...,...,...,...
232267,2026,181,19,47
232268,2026,181,20,47
232269,2026,181,21,33
232270,2026,181,22,33


In [14]:
omni_df_combined = pd.merge(omni_df, kp_df, on=['year', 'day', 'hour'], how='left')
omni_df_combined

,year,day,hour,minute,mag_avg_nt,bx_gsm_nt,by_gsm_nt,bz_gsm_nt,flow_speed_km_s,proton_density_n_cc,day_cos,day_sin,hour_cos,hour_sin,minute_cos,minute_sin,kp_10
0,2000,1,0,0,6.82,-5.94,0.27,-0.08,NaN,NaN,0.999852,0.017213,1.000000,0.000000,1.000000,0.000000,53
1,2000,1,0,1,6.99,-5.88,1.95,1.08,664.7,3.12,0.999852,0.017213,1.000000,0.000000,0.994522,0.104528,53
2,2000,1,0,2,6.99,-5.71,2.74,2.24,663.2,3.24,0.999852,0.017213,1.000000,0.000000,0.978148,0.207912,53
3,2000,1,0,3,6.83,-5.33,3.18,2.78,662.2,3.11,0.999852,0.017213,1.000000,0.000000,0.951057,0.309017,53
4,2000,1,0,4,NaN,NaN,NaN,NaN,NaN,NaN,0.999852,0.017213,1.000000,0.000000,0.913545,0.406737,53
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13936315,2026,181,23,55,17.93,0.28,-16.97,5.79,432.8,18.85,-0.999667,0.025818,0.965926,-0.258819,0.866025,-0.500000,33
13936316,2026,181,23,56,18.15,0.01,-17.13,6.01,432.3,16.93,-0.999667,0.025818,0.965926,-0.258819,0.913545,-0.406737,33
13936317,2026,181,23,57,18.12,0.16,-16.98,6.31,432.1,16.35,-0.999667,0.025818,0.965926,-0.258819,0.951057,-0.309017,33
13936318,2026,181,23,58,17.98,0.36,-16.55,7.02,431.1,17.80,-0.999667,0.025818,0.965926,-0.258819,0.978148,-0.207912,33


In [15]:
omni_df_combined.to_parquet("data/Processed/omni_minute.parquet")

In [16]:
omni_df_combined.isna().sum()

year                         0
day                          0
hour                         0
minute                       0
mag_avg_nt              995865
bx_gsm_nt               995865
by_gsm_nt               995865
bz_gsm_nt               995865
flow_speed_km_s        3041365
proton_density_n_cc    3041362
day_cos                      0
day_sin                      0
hour_cos                     0
hour_sin                     0
minute_cos                   0
minute_sin                   0
kp_10                        0
dtype: int64

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a18bf7d-431c-4909-af8a-a54a62228a78' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>